In [3]:
# Install PySpark

!pip install pyspark==3.5.1

In [4]:
# Import PySpark and check version

import pyspark

print(pyspark.__version__)

3.5.1


In [5]:
# Import SparkSession

from pyspark.sql import SparkSession

# Create Spark Session

spark = SparkSession.builder \
    .appName("Assignment 5") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [7]:
# Create sample employee dataset

data = [
    (101, "Arjun", 23, "IT", 52000),
    (102, "Meera", 28, "HR", 47000),
    (103, "Kabir", None, "Finance", 65000),
    (104, "Sneha", 30, "Marketing", None),
    (105, "Dev", 26, None, 58000),
    (106, "Ananya", 24, "HR", 49000),
    (107, "Yash", None, "IT", 62000),
    (108, "Ritika", 27, "Finance", 70000),
    (102, "Meera", 28, "HR", 47000),   # Duplicate Record
    (109, "Kunal", 33, "IT", 85000)
]

# Define column names

columns = ["Emp_ID", "Name", "Age", "Department", "Salary"]

# Create DataFrame

df = spark.createDataFrame(data, columns)

In [8]:
df.show()

+------+------+----+----------+------+
|Emp_ID|  Name| Age|Department|Salary|
+------+------+----+----------+------+
|   101| Arjun|  23|        IT| 52000|
|   102| Meera|  28|        HR| 47000|
|   103| Kabir|NULL|   Finance| 65000|
|   104| Sneha|  30| Marketing|  NULL|
|   105|   Dev|  26|      NULL| 58000|
|   106|Ananya|  24|        HR| 49000|
|   107|  Yash|NULL|        IT| 62000|
|   108|Ritika|  27|   Finance| 70000|
|   102| Meera|  28|        HR| 47000|
|   109| Kunal|  33|        IT| 85000|
+------+------+----+----------+------+



In [9]:
# Display Schema

df.printSchema()

root
 |-- Emp_ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: long (nullable = true)



In [10]:
# Count null values in each column

from pyspark.sql.functions import col, when, count

# Count null values in each column

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+----+---+----------+------+
|Emp_ID|Name|Age|Department|Salary|
+------+----+---+----------+------+
|     0|   0|  2|         1|     1|
+------+----+---+----------+------+



In [11]:
# Replace null values

df = df.fillna({
    "Age": 0,
    "Department": "Unknown",
    "Salary": 0
})

# Display updated DataFrame

df.show()

+------+------+---+----------+------+
|Emp_ID|  Name|Age|Department|Salary|
+------+------+---+----------+------+
|   101| Arjun| 23|        IT| 52000|
|   102| Meera| 28|        HR| 47000|
|   103| Kabir|  0|   Finance| 65000|
|   104| Sneha| 30| Marketing|     0|
|   105|   Dev| 26|   Unknown| 58000|
|   106|Ananya| 24|        HR| 49000|
|   107|  Yash|  0|        IT| 62000|
|   108|Ritika| 27|   Finance| 70000|
|   102| Meera| 28|        HR| 47000|
|   109| Kunal| 33|        IT| 85000|
+------+------+---+----------+------+



In [12]:
# Count before removing duplicates

print("Before:", df.count())

# Remove duplicates

df = df.dropDuplicates()

# Count after removing duplicates

print("After:", df.count())

Before: 10
After: 9


In [13]:
# Create a new DataFrame

df2 = df.filter(col("Age") > 25)

# Original DataFrame remains unchanged

print("Original Count:", df.count())
print("Filtered Count:", df2.count())

Original Count: 9
Filtered Count: 5


In [14]:
# Filter employees age greater than 25

df.filter(df.Age > 25).show()

+------+------+---+----------+------+
|Emp_ID|  Name|Age|Department|Salary|
+------+------+---+----------+------+
|   102| Meera| 28|        HR| 47000|
|   104| Sneha| 30| Marketing|     0|
|   105|   Dev| 26|   Unknown| 58000|
|   108|Ritika| 27|   Finance| 70000|
|   109| Kunal| 33|        IT| 85000|
+------+------+---+----------+------+



In [15]:
# Filter employees from IT department

df.filter(df.Department == "IT").show()

+------+-----+---+----------+------+
|Emp_ID| Name|Age|Department|Salary|
+------+-----+---+----------+------+
|   101|Arjun| 23|        IT| 52000|
|   107| Yash|  0|        IT| 62000|
|   109|Kunal| 33|        IT| 85000|
+------+-----+---+----------+------+



In [16]:
# Create Bonus column (10% of Salary)

from pyspark.sql.functions import col

df = df.withColumn(
    "Bonus",
    col("Salary") * 0.10
)

df.show()

+------+------+---+----------+------+------+
|Emp_ID|  Name|Age|Department|Salary| Bonus|
+------+------+---+----------+------+------+
|   103| Kabir|  0|   Finance| 65000|6500.0|
|   102| Meera| 28|        HR| 47000|4700.0|
|   104| Sneha| 30| Marketing|     0|   0.0|
|   105|   Dev| 26|   Unknown| 58000|5800.0|
|   101| Arjun| 23|        IT| 52000|5200.0|
|   106|Ananya| 24|        HR| 49000|4900.0|
|   108|Ritika| 27|   Finance| 70000|7000.0|
|   107|  Yash|  0|        IT| 62000|6200.0|
|   109| Kunal| 33|        IT| 85000|8500.0|
+------+------+---+----------+------+------+



In [17]:
# Calculate average salary department-wise

from pyspark.sql.functions import avg

df.groupBy("Department") \
  .agg(avg("Salary").alias("Average_Salary")) \
  .show()

+----------+-----------------+
|Department|   Average_Salary|
+----------+-----------------+
|        HR|          48000.0|
|   Finance|          67500.0|
|   Unknown|          58000.0|
| Marketing|              0.0|
|        IT|66333.33333333333|
+----------+-----------------+



In [18]:
# Calculate total salary department-wise

from pyspark.sql.functions import sum

df.groupBy("Department") \
  .agg(sum("Salary").alias("Total_Salary")) \
  .show()

+----------+------------+
|Department|Total_Salary|
+----------+------------+
|        HR|       96000|
|   Finance|      135000|
|   Unknown|       58000|
| Marketing|           0|
|        IT|      199000|
+----------+------------+



In [19]:
# Count employees department-wise

from pyspark.sql.functions import count

df.groupBy("Department") \
  .agg(count("*").alias("Employee_Count")) \
  .show()

+----------+--------------+
|Department|Employee_Count|
+----------+--------------+
|        HR|             2|
|   Finance|             2|
|   Unknown|             1|
| Marketing|             1|
|        IT|             3|
+----------+--------------+



In [20]:
# Find minimum salary

from pyspark.sql.functions import min

df.select(
    min("Salary").alias("Minimum_Salary")
).show()

+--------------+
|Minimum_Salary|
+--------------+
|             0|
+--------------+



In [21]:
# Find maximum salary

from pyspark.sql.functions import max

df.select(
    max("Salary").alias("Maximum_Salary")
).show()

+--------------+
|Maximum_Salary|
+--------------+
|         85000|
+--------------+



In [22]:
# Rename Salary column

df = df.withColumnRenamed(
    "Salary",
    "Monthly_Salary"
)

df.show()

+------+------+---+----------+--------------+------+
|Emp_ID|  Name|Age|Department|Monthly_Salary| Bonus|
+------+------+---+----------+--------------+------+
|   103| Kabir|  0|   Finance|         65000|6500.0|
|   102| Meera| 28|        HR|         47000|4700.0|
|   104| Sneha| 30| Marketing|             0|   0.0|
|   105|   Dev| 26|   Unknown|         58000|5800.0|
|   101| Arjun| 23|        IT|         52000|5200.0|
|   106|Ananya| 24|        HR|         49000|4900.0|
|   108|Ritika| 27|   Finance|         70000|7000.0|
|   107|  Yash|  0|        IT|         62000|6200.0|
|   109| Kunal| 33|        IT|         85000|8500.0|
+------+------+---+----------+--------------+------+



In [23]:
# Convert columns into integer datatype

df = df.withColumn(
    "Age",
    col("Age").cast("integer")
)

df = df.withColumn(
    "Monthly_Salary",
    col("Monthly_Salary").cast("integer")
)

df.printSchema()

root
 |-- Emp_ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Department: string (nullable = false)
 |-- Monthly_Salary: integer (nullable = false)
 |-- Bonus: double (nullable = false)



In [24]:
# Sort employees by salary

df.orderBy(
    col("Monthly_Salary").desc()
).show()

+------+------+---+----------+--------------+------+
|Emp_ID|  Name|Age|Department|Monthly_Salary| Bonus|
+------+------+---+----------+--------------+------+
|   109| Kunal| 33|        IT|         85000|8500.0|
|   108|Ritika| 27|   Finance|         70000|7000.0|
|   103| Kabir|  0|   Finance|         65000|6500.0|
|   107|  Yash|  0|        IT|         62000|6200.0|
|   105|   Dev| 26|   Unknown|         58000|5800.0|
|   101| Arjun| 23|        IT|         52000|5200.0|
|   106|Ananya| 24|        HR|         49000|4900.0|
|   102| Meera| 28|        HR|         47000|4700.0|
|   104| Sneha| 30| Marketing|             0|   0.0|
+------+------+---+----------+--------------+------+



In [25]:
# Average salary department-wise

from pyspark.sql.functions import avg

dept = df.groupBy("Department") \
         .agg(
             avg("Monthly_Salary").alias("AvgSalary")
         )

# Display departments with average salary > 50000

dept.filter(
    col("AvgSalary") > 50000
).show()

+----------+-----------------+
|Department|        AvgSalary|
+----------+-----------------+
|   Finance|          67500.0|
|   Unknown|          58000.0|
|        IT|66333.33333333333|
+----------+-----------------+



In [26]:
# Complete ETL Pipeline

result = (
    df.dropDuplicates()
      .fillna({
          "Age": 0,
          "Department": "Unknown",
          "Monthly_Salary": 0
      })
      .filter(col("Age") > 20)
      .groupBy("Department")
      .agg(
          avg("Monthly_Salary").alias("AvgSalary"),
          count("*").alias("EmployeeCount")
      )
)

result.show()

+----------+---------+-------------+
|Department|AvgSalary|EmployeeCount|
+----------+---------+-------------+
|        HR|  48000.0|            2|
|   Finance|  70000.0|            1|
|   Unknown|  58000.0|            1|
| Marketing|      0.0|            1|
|        IT|  68500.0|            2|
+----------+---------+-------------+



In [27]:
# Save cleaned data into CSV

df.coalesce(1) \
  .write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("cleaned_employee_data")